In [1]:
import os
from urllib.parse import quote_plus
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
pw = quote_plus(os.getenv("DB_PASSWORD"))
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{pw}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
clean = Path("../data/clean"
             )
with engine.connect() as conn:
    print(conn.execute(text("SELECT DATABASE()")).scalar())

funnel_db


In [3]:
def prep(df):
    """Make a DataFrame safe for MySQL: UTC timestamps without a time zone, booleans as 0/1."""
    df=df.copy()
    for c in df.columns:
        dtype = str(df[c].dtype)
        if dtype.startswith("datetime64") and df[c].dt.tz is not None:
            df[c] = df[c].dt.tz_convert("UTC").dt.tz_localize(None)
        elif dtype in ("bool", "boolean"):
            df[c] = df[c].astype("int8")
    return df

for table in ["users","sessions","events","orders"]:
    df = prep(pd.read_parquet(clean / f"{table}.parquet"))
    with engine.begin() as conn:
        conn.execute(text(f"TRUNCATE TABLE {table}"))
    df.to_sql(table, engine, if_exists="append",index = False,
              chunksize=5000, method="multi")
    print(table, "loaded:", len(df))
    del df

DatabaseError: Execution failed on sql 'INSERT INTO users (full_visitor_id, first_visit_number, first_session_date, cohort_month, is_true_new, first_channel, first_category, first_country, has_inconsistent_visit_number, in_cohort_analysis, is_heavy_visitor, valid_orders, is_heavy_buyer) VALUES (:full_visitor_id_m0, :first_visit_number_m0, :first_session_date_m0, :cohort_month_m0, :is_true_new_m0, :first_channel_m0, :first_category_m0, :first_country_m0, :has_inconsistent_visit_number_m0, :in_cohort_analysis_m0, :is_heavy_visitor_m0, :valid_orders_m0, :is_heavy_buyer_m0), (:full_visitor_id_m1, :first_visit_number_m1, :first_session_date_m1, :cohort_month_m1, :is_true_new_m1, :first_channel_m1, :first_category_m1, :first_country_m1, :has_inconsistent_visit_number_m1, :in_cohort_analysis_m1, :is_heavy_visitor_m1, :valid_orders_m1, :is_heavy_buyer_m1), (:full_visitor_id_m2, :first_visit_number_m2, :first_session_date_m2, :cohort_month_m2, :is_true_new_m2, :first_channel_m2, :first_category